In [135]:
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import ks_2samp, wasserstein_distance, gaussian_kde
from scipy.spatial.distance import jensenshannon
from sklearn.preprocessing import StandardScaler
import statsmodels.formula.api as smf

In [136]:
final_df = pd.read_csv('master_df_final.csv')
questions = final_df['Question'].unique()

participant_data = final_df[final_df['Question'] == 'Q3F']
participant_data = participant_data[['ID','BeforeNumber','RACE','AGEBRACKET','EDUCATION','PARTYBEFORE','GENDER']]

In [137]:
# calculate the percentages of each demographic for each colum
for col in ['BeforeNumber','RACE','AGEBRACKET','EDUCATION','PARTYBEFORE','GENDER']:
    print(f"Percentages for {col}:")
    valcounts = participant_data[col].value_counts(normalize=True)
    diversity_num = sum(valcounts**2)
    print(f"Diversity index for {col}: {diversity_num:.4f}")
    print(valcounts * 10)
    # calculate diversity index for each column
    print("\n")

# BeforeNumber = 4 with 0, 1 with 1, 1 with 2 or 3, 2 with 4 or 5, 1 with 6, 7 or 8, 1 with 8 or 10
# Race= 6 white, 2 black, 1 asian, 1 hispanic
# AgeBracket = 2 with 18-29, 2 with 30-44, 3 with 45-60, 3 with 60+
# Education = 1 with high school graduate, 4 with some college, 5 with bachelor's degree or higher
# PartyBefore = 3 with democrat, 3 with republican, 4 with independent

Percentages for BeforeNumber:
Diversity index for BeforeNumber: 0.2302
BeforeNumber
0     4.340344
5     1.472275
2     0.669216
1     0.592734
3     0.592734
10    0.516252
7     0.497132
8     0.325048
4     0.305927
6     0.267686
77    0.191205
9     0.191205
98    0.038241
Name: proportion, dtype: float64


Percentages for RACE:
Diversity index for RACE: 0.4491
RACE
White, non-Hispanic    6.386233
Black, non-Hispanic    1.644359
Hispanic               1.013384
Asian, non-Hispanic    0.497132
2+, non-Hispanic       0.382409
Other, non-Hispanic    0.076482
Name: proportion, dtype: float64


Percentages for AGEBRACKET:
Diversity index for AGEBRACKET: 0.2662
AGEBRACKET
60+      3.384321
45-59    2.772467
30-44    2.141491
18-29    1.701721
Name: proportion, dtype: float64


Percentages for EDUCATION:
Diversity index for EDUCATION: 0.4220
EDUCATION
BA or above                  5.124283
Some college                 3.900574
HS graduate or equivalent    0.841300
No HS diploma            

In [138]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# df: one row per participant
# columns: ID, gender, party, ideology_score, baseline_q, age, ...

# 1) Encode party etc. numerically
df_ = participant_data.copy()

# 2) Choose matching covariates (excluding gender)
match_cols = ['BeforeNumber','RACE','AGEBRACKET','EDUCATION','PARTYBEFORE']

X = pd.get_dummies(df_[match_cols])  # One-hot encode categorical variables
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3) Build small blocks via clustering (e.g. blocks of 4)
#    n_clusters ~ N / block_size
block_size = 2
n_clusters = len(df_) // block_size

kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
df_['block'] = kmeans.fit_predict(X_scaled)
df_.sort_values('block', inplace=True)

In [139]:
# drop dublicate blocks (keep first)
df_ = df_.drop_duplicates(subset=['block',"GENDER"])

In [140]:
pairs = df_['block'].value_counts().reset_index(name='count')
pairs = pairs[pairs['count'] == 2] # Only keep blocks with exactly 2 participants
matched_pairs = []
for block in pairs['block']:
    pair = df_[df_['block'] == block]
    if len(pair) == 2:
        matched_pairs.append(pair)
matched_df = pd.concat(matched_pairs)

In [141]:
matched_df_ppl = matched_df[['BeforeNumber','RACE','AGEBRACKET','EDUCATION','PARTYBEFORE','block']].drop_duplicates()
matched_df_ppl.to_csv('matched_pairs.csv', index=False)

In [142]:
ids = [33,70,3,21,2,7,41,61,58,26]
# take away one from all ids
for i in range(10):
    ids[i] = ids[i] - 1
matched_df_ppl = matched_df_ppl.iloc[ids,:]

In [143]:
chosen = matched_df_ppl['block'].to_list()

In [144]:
for col in ['BeforeNumber','RACE','AGEBRACKET','EDUCATION','PARTYBEFORE']:
    print(f"Percentages for {col}:")
    valcounts = matched_df_ppl[col].value_counts(normalize=True)
    diversity_num = sum(valcounts**2)
    print(f"Diversity index for {col}: {diversity_num:.4f}")
    print(valcounts * 10)
    # calculate diversity index for each column
    print("\n")


Percentages for BeforeNumber:
Diversity index for BeforeNumber: 0.1600
BeforeNumber
0     3.0
2     1.0
10    1.0
3     1.0
9     1.0
6     1.0
7     1.0
1     1.0
Name: proportion, dtype: float64


Percentages for RACE:
Diversity index for RACE: 0.5400
RACE
White, non-Hispanic    7.0
Black, non-Hispanic    2.0
Asian, non-Hispanic    1.0
Name: proportion, dtype: float64


Percentages for AGEBRACKET:
Diversity index for AGEBRACKET: 0.2800
AGEBRACKET
60+      4.0
18-29    2.0
45-59    2.0
30-44    2.0
Name: proportion, dtype: float64


Percentages for EDUCATION:
Diversity index for EDUCATION: 0.4200
EDUCATION
BA or above                  5.0
Some college                 4.0
HS graduate or equivalent    1.0
Name: proportion, dtype: float64


Percentages for PARTYBEFORE:
Diversity index for PARTYBEFORE: 0.4200
PARTYBEFORE
Independent    5.0
Republican     4.0
Democrat       1.0
Name: proportion, dtype: float64




In [145]:
experiment_participants = matched_df[matched_df['block'].isin(chosen)].sort_values(["block","GENDER"], ascending = [True, True]).reset_index()

In [150]:
import json

# make a json object with the matched pairs
matched_pairs_json = []

for i in range(20):
    group = i % 2
    json_object = {
        "id" : int(experiment_participants.iloc[i,1]),
        "group" : group,
        "demographics" : {
            "RACE" : experiment_participants.iloc[i,3],
            "AGEBRACKET" : experiment_participants.iloc[i,4],
            "EDUCATION" : experiment_participants.iloc[i,5],
            "PARTYBEFORE" : experiment_participants.iloc[i,6],
            "GENDER" : experiment_participants.iloc[i,7],
            "AGE" : 20 # not used in the data, so just put 20
        },
        "questions" : {
            "Q3F" : int(experiment_participants.iloc[i,2])
        },
        "attitudes": { # again also placeholder as we don't use this
            "Democracy System Approval": "In the middle",
            "Restrictive Immigration Attitude": "Extremely oppose",
            "Immigration Policy Attitude": "In the middle",
            "Immigration Support Attitude": "Extremely oppose",
            "Pro-Climate Regulation Attitude": "Strongly favor",
            "Climate Policy Attitude": "In the middle",
            "Pro-Fossil Fuel Attitude": "In the middle",
            "Tax Policy Attitude": "Extremely oppose",
            "Progressive Tax/Support Attitude": "Extremely oppose",
            "Tax Reduction/Conservative Attitude": "Strongly favor",
            "Healthcare System Reform Attitude": "Extremely oppose",
            "Healthcare Expansion Attitude": "Strongly favor",
            "Healthcare Policy Attitude": "Extremely oppose",
            "Global Politics Attitude": "Extremely oppose"
        }}
    matched_pairs_json.append(json_object)

experiment_participants.sort_values(["GENDER","block"], ascending = [True, True])
for i in range(20):
    group = i % 4
    if group % 3 == 0:
        group = 2
    else:
         group = 3
    json_object = {
        "id" : int(str(experiment_participants.iloc[i,1])),
        "group" : group,
        "demographics" : {
            "RACE" : experiment_participants.iloc[i,3],
            "AGEBRACKET" : experiment_participants.iloc[i,4],
            "EDUCATION" : experiment_participants.iloc[i,5],
            "PARTYBEFORE" : experiment_participants.iloc[i,6],
            "GENDER" : experiment_participants.iloc[i,7],
            "AGE" : 20 # not used in the data, so just put 20
        },
        "questions" : {
            "Q3F" : int(str(experiment_participants.iloc[i,2]))
        },
        "attitudes": { # again also placeholder as we don't use this
            "Democracy System Approval": "In the middle",
            "Restrictive Immigration Attitude": "Extremely oppose",
            "Immigration Policy Attitude": "In the middle",
            "Immigration Support Attitude": "Extremely oppose",
            "Pro-Climate Regulation Attitude": "Strongly favor",
            "Climate Policy Attitude": "In the middle",
            "Pro-Fossil Fuel Attitude": "In the middle",
            "Tax Policy Attitude": "Extremely oppose",
            "Progressive Tax/Support Attitude": "Extremely oppose",
            "Tax Reduction/Conservative Attitude": "Strongly favor",
            "Healthcare System Reform Attitude": "Extremely oppose",
            "Healthcare Expansion Attitude": "Strongly favor",
            "Healthcare Policy Attitude": "Extremely oppose",
            "Global Politics Attitude": "Extremely oppose"
        }}
    matched_pairs_json.append(json_object)   

with open("matched_pairs.json", 'w', encoding='utf-8') as f:
        json.dump(matched_pairs_json, f, ensure_ascii=False, indent=4)
            

In [147]:
matched_pairs_json

[{'id': 348,
  'group': 0,
  'demographics': {'RACE': 'White, non-Hispanic',
   'AGEBRACKET': '60+',
   'EDUCATION': 'BA or above',
   'PARTYBEFORE': 'Independent',
   'GENDER': 'Female',
   'AGE': 20},
  'questions': {'Q3F': 7},
  'attitudes': {'Democracy System Approval': 'In the middle',
   'Restrictive Immigration Attitude': 'Extremely oppose',
   'Immigration Policy Attitude': 'In the middle',
   'Immigration Support Attitude': 'Extremely oppose',
   'Pro-Climate Regulation Attitude': 'Strongly favor',
   'Climate Policy Attitude': 'In the middle',
   'Pro-Fossil Fuel Attitude': 'In the middle',
   'Tax Policy Attitude': 'Extremely oppose',
   'Progressive Tax/Support Attitude': 'Extremely oppose',
   'Tax Reduction/Conservative Attitude': 'Strongly favor',
   'Healthcare System Reform Attitude': 'Extremely oppose',
   'Healthcare Expansion Attitude': 'Strongly favor',
   'Healthcare Policy Attitude': 'Extremely oppose',
   'Global Politics Attitude': 'Extremely oppose'}},
 {'id':

In [148]:
{
        "id": 0,
        "group": 1,
        "demographics": {
            "RACE": "White, non-Hispanic",
            "AGEBRACKET": "18-29",
            "EDUCATION": "Some college",
            "PARTYBEFORE": "Independent",
            "GENDER": "Male",
            "AGE": 21
        },
        "questions": {
            "Q1": 5,
            "Q2A": 0,
            "Q2B": 10,
            "Q2C": 0,
            "Q2D": 5,
            "Q2E": 5,
            "Q2F": 5,
            "Q2G": 10,
            "Q2H": 10,
            "Q2I": 10,
            "Q3A": 10,
            "Q3B": 10,
            "Q3C": 10,
            "Q3D": 10,
            "Q3E": 5,
            "Q3F": 0,
            "Q3G": 5,
            "Q3H": 5,
            "Q4A": 0,
            "Q4B": 0,
            "Q4C": 10,
            "Q4D": 0,
            "Q4E": 0,
            "Q4F": 10,
            "Q4G": 0,
            "Q4H": 0,
            "Q4I": 0,
            "Q4J": 0,
            "Q5A": 10,
            "Q5B": 0,
            "Q5C": 0,
            "Q5D": 0,
            "Q5E": 10,
            "Q5F": 10,
            "Q5G": 10,
            "Q5H": 0,
            "Q5I": 10,
            "Q5J": 5,
            "Q6A": 10,
            "Q6B": 0,
            "Q6C": 0,
            "Q6D": 10,
            "Q6E": 0,
            "Q6F": 0,
            "Q6G": 10,
            "Q6H": 10,
            "Q6I": 0,
            "Q6J": 0
        },
        "attitudes": {
            "Democracy System Approval": "In the middle",
            "Restrictive Immigration Attitude": "Extremely oppose",
            "Immigration Policy Attitude": "In the middle",
            "Immigration Support Attitude": "Extremely oppose",
            "Pro-Climate Regulation Attitude": "Strongly favor",
            "Climate Policy Attitude": "In the middle",
            "Pro-Fossil Fuel Attitude": "In the middle",
            "Tax Policy Attitude": "Extremely oppose",
            "Progressive Tax/Support Attitude": "Extremely oppose",
            "Tax Reduction/Conservative Attitude": "Strongly favor",
            "Healthcare System Reform Attitude": "Extremely oppose",
            "Healthcare Expansion Attitude": "Strongly favor",
            "Healthcare Policy Attitude": "Extremely oppose",
            "Global Politics Attitude": "Extremely oppose"
        }
    },

({'id': 0,
  'group': 1,
  'demographics': {'RACE': 'White, non-Hispanic',
   'AGEBRACKET': '18-29',
   'EDUCATION': 'Some college',
   'PARTYBEFORE': 'Independent',
   'GENDER': 'Male',
   'AGE': 21},
  'questions': {'Q1': 5,
   'Q2A': 0,
   'Q2B': 10,
   'Q2C': 0,
   'Q2D': 5,
   'Q2E': 5,
   'Q2F': 5,
   'Q2G': 10,
   'Q2H': 10,
   'Q2I': 10,
   'Q3A': 10,
   'Q3B': 10,
   'Q3C': 10,
   'Q3D': 10,
   'Q3E': 5,
   'Q3F': 0,
   'Q3G': 5,
   'Q3H': 5,
   'Q4A': 0,
   'Q4B': 0,
   'Q4C': 10,
   'Q4D': 0,
   'Q4E': 0,
   'Q4F': 10,
   'Q4G': 0,
   'Q4H': 0,
   'Q4I': 0,
   'Q4J': 0,
   'Q5A': 10,
   'Q5B': 0,
   'Q5C': 0,
   'Q5D': 0,
   'Q5E': 10,
   'Q5F': 10,
   'Q5G': 10,
   'Q5H': 0,
   'Q5I': 10,
   'Q5J': 5,
   'Q6A': 10,
   'Q6B': 0,
   'Q6C': 0,
   'Q6D': 10,
   'Q6E': 0,
   'Q6F': 0,
   'Q6G': 10,
   'Q6H': 10,
   'Q6I': 0,
   'Q6J': 0},
  'attitudes': {'Democracy System Approval': 'In the middle',
   'Restrictive Immigration Attitude': 'Extremely oppose',
   'Immigration Polic